In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from PIL import Image
import torchvision
import numpy as np
import os

In [3]:
data_train = torchvision.datasets.MNIST(root='data', train=True, download=True)
data_test = torchvision.datasets.MNIST(root='data', train=False, download=True)

print(data_train[8])

(<PIL.Image.Image image mode=L size=28x28 at 0x29EB26D4690>, 1)


In [4]:
MNIST_ROOT = './mnist_folder'

def save_mnist_to_folder(mnist_dataset, split_name, root_dir):
    split_dir = os.path.join(root_dir, split_name)

    for label in range(10):
        os.makedirs(os.path.join(split_dir, str(label)), exist_ok=True)

    for idx, (image, label) in enumerate(mnist_dataset):
        path = os.path.join(split_dir, str(label), f"img_{idx:05d}.png")
        image.save(path)

save_mnist_to_folder(data_train, 'train', MNIST_ROOT)
save_mnist_to_folder(data_test, 'test', MNIST_ROOT)

In [13]:
class MNISTDataset(Dataset):
    def __init__(self, root_dir, transfoms=None):
        self.root_dir = root_dir
        self.transfoms = transfoms

        self.image_path = []
        self.labels = []

        class_folders = sorted(os.listdir(root_dir))

        self.class_to_idx = {
            folder_name: idx for idx, folder_name in enumerate(class_folders)
        }

        for folder_name in class_folders:
            folder_path = os.path.join(root_dir, folder_name)

            label = self.class_to_idx[folder_name]

            for filename in os.listdir(folder_path):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_path.append(os.path.join(folder_path, filename))
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_path)

    def __getitem__(self, idx):
        img_path = self.image_path[idx]
        label = self.labels[idx]

        image = Image.open(img_path)

        if self.transfoms:
            image = self.transfoms(image)

        return image, torch.tensor(label, dtype=torch.long)

In [14]:
train_dir = os.path.join(MNIST_ROOT, 'train')
mnist_train_dataset = MNISTDataset(root_dir=train_dir)

image, label = mnist_train_dataset[1]

print(f'Image: {image.size}, label: {label.item()}')

Image: (28, 28), label: 0
